## Mask2Former-Swin-Large Panoptic Segmentation

In [ ]:
import pandas  as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import torch
import torch.nn as nn
import torch.optim as optim
import sys
import glob
import json
from PIL import Image as PILImage
from collections import defaultdict
from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [ ]:
# Mask2Former-Swin-Large (Hugging Face)
# Uses transformers library to load facebook/mask2former-swin-large-coco-panoptic

USE_HF_SWIN_LARGE = False
mask2former_model = None
processor = None

try:
    model_id = "facebook/mask2former-swin-large-coco-panoptic"
    print(f"Loading {model_id} from Hugging Face...")
    
    processor = Mask2FormerImageProcessor.from_pretrained(model_id)
    # Use safetensors to avoid torch.load CVE-2025-32434 restriction (requires torch>=2.6)
    mask2former_model = Mask2FormerForUniversalSegmentation.from_pretrained(
        model_id, use_safetensors=True
    )
    
    mask2former_model.to(device)
    mask2former_model.eval()
    
    USE_HF_SWIN_LARGE = True
    print(f"Successfully loaded {model_id}")
    print(f"Running on: {device}")
    
    total_params = sum(p.numel() for p in mask2former_model.parameters())
    print(f"Total parameters: {total_params:,}")

except Exception as e:
    print(f"Failed to load Hugging Face model: {e}")

In [ ]:
# Run Mask2Former-Swin-Large predictions on ACDC dataset

# ACDC dataset path
acdc_path = r"C:\desktop 2\project\datasets\ACDC"

# Get all images from the ACDC dataset
image_files = glob.glob(os.path.join(acdc_path, "*.jpg")) + \
              glob.glob(os.path.join(acdc_path, "*.png"))

print(f"Found {len(image_files)} images in ACDC dataset")

# COCO panoptic class names (Standard COCO list)
COCO_PANOPTIC_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck',
    'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench',
    'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra',
    'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee',
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup',
    'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange',
    'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
    'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse',
    'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush', 'banner', 'blanket', 'bridge', 'cardboard', 'counter', 'curtain',
    'door-stuff', 'floor-wood', 'flower', 'fruit', 'gravel', 'house', 'light',
    'mirror-stuff', 'net', 'pillow', 'platform', 'playingfield', 'railroad',
    'river', 'road', 'roof', 'sand', 'sea', 'shelf', 'snow', 'stairs', 'tent',
    'towel', 'wall-brick', 'wall-stone', 'wall-tile', 'wall-wood', 'water',
    'window-blind', 'window', 'tree', 'fence', 'ceiling', 'sky', 'cabinet',
    'table', 'floor', 'pavement', 'mountain', 'grass', 'dirt', 'paper', 'food',
    'building', 'rock', 'wall', 'rug'
]

def render_panoptic(panoptic_seg, segments_info, img_np, img_name, filter_acdc=False):
    """Visualize panoptic output with labels, bounding boxes, and scores."""
    colored_seg = np.zeros_like(img_np)
    np.random.seed(42)
    colors = np.random.randint(0, 255, (256, 3), dtype=np.uint8)

    class_instance_counts = {}
    segment_labels = []
    
    # Use all segments (show all COCO classes)
    filtered_segments = segments_info

    for seg in filtered_segments:
        seg_id = seg['id']
        label_id = seg['label_id']

        class_name = COCO_PANOPTIC_CLASSES[label_id] if 0 <= label_id < len(COCO_PANOPTIC_CLASSES) else f"class_{label_id}"

        class_instance_counts[class_name] = class_instance_counts.get(class_name, 0) + 1
        instance_num = class_instance_counts[class_name]
        instance_label = f"{class_name}{instance_num}"

        mask = panoptic_seg == seg_id
        colored_seg[mask] = colors[seg_id % 256]

        if np.any(mask):
            y_coords, x_coords = np.where(mask)
            centroid_x = int(np.mean(x_coords))
            centroid_y = int(np.mean(y_coords))
            
            x_min, x_max = np.min(x_coords), np.max(x_coords)
            y_min, y_max = np.min(y_coords), np.max(y_coords)
            
            segment_labels.append({
                'label': instance_label,
                'x': centroid_x,
                'y': centroid_y,
                'bbox': (x_min, y_min, x_max, y_max),
                'color': colors[seg_id % 256].tolist()
            })
        
    alpha = 0.5
    blended = (img_np * (1 - alpha) + colored_seg * alpha).astype(np.uint8)
    blended_with_labels = cv2.cvtColor(blended, cv2.COLOR_RGB2BGR)

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.6
    font_thickness = 2

    for label_info in segment_labels:
        label_text = f"{label_info['label']}"
        x, y = label_info['x'], label_info['y']
        
        x1, y1, x2, y2 = label_info['bbox']
        box_color = label_info['color']
        cv2.rectangle(blended_with_labels, (x1, y1), (x2, y2), box_color, 2)
        
        (text_width, text_height), baseline = cv2.getTextSize(label_text, font, font_scale, font_thickness)
        
        x = max(5, min(x - text_width // 2, blended_with_labels.shape[1] - text_width - 5))
        y = max(text_height + 10, min(y, blended_with_labels.shape[0] - 5))
        
        cv2.rectangle(blended_with_labels, (x - 2, y - text_height - 5), (x + text_width + 2, y + 5), (0, 0, 0), -1)
        cv2.putText(blended_with_labels, label_text, (x, y), font, font_scale, (255, 255, 255), font_thickness)

    blended_with_labels = cv2.cvtColor(blended_with_labels, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    axes[0].imshow(img_np)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    axes[1].imshow(colored_seg)
    axes[1].set_title("Panoptic Segmentation")
    axes[1].axis('off')
    axes[2].imshow(blended_with_labels)
    axes[2].set_title("Labeled Segmentation + BBox")
    axes[2].axis('off')
    plt.suptitle(f"Mask2Former-Swin-Large (COCO Panoptic)\n{img_name}", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    return len(filtered_segments)

if USE_HF_SWIN_LARGE:
    print(f"\nShowing all {len(COCO_PANOPTIC_CLASSES)} COCO panoptic classes")
    
    # Limit display to 10 images
    MAX_DISPLAY = 10
    display_files = image_files[:MAX_DISPLAY]
    print(f"Displaying {len(display_files)} of {len(image_files)} images")
    
    for img_path in display_files:
        print(f"\n{'='*60}")
        print(f"Processing: {os.path.basename(img_path)}")
        print('='*60)

        image = PILImage.open(img_path).convert("RGB")
        img_np = np.array(image)

        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = mask2former_model(**inputs)

        result = processor.post_process_panoptic_segmentation(
            outputs, target_sizes=[image.size[::-1]]
        )[0]

        panoptic_seg = result["segmentation"].cpu().numpy()
        segments_info = result["segments_info"]

        # Show all COCO classes
        seg_count = render_panoptic(panoptic_seg, segments_info, img_np, os.path.basename(img_path), filter_acdc=False)
        print(f"Objects detected: {seg_count} segments")

else:
    print("\nMask2Former-Swin-Large inference unavailable.")
    print("Please ensure transformers is installed.")
    
    for img_path in image_files[:3]:
        img = cv2.imread(img_path)
        if img is None: continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 8))
        plt.imshow(img_rgb)
        plt.title(f"ACDC Dataset: {os.path.basename(img_path)}")
        plt.axis('off')

        plt.show()
        print("="*60)

print("Mask2Former-Swin-Large processing complete!")
print("\n" + "="*60)

In [ ]:
# COCO val2017 Evaluation Function (with size classification)
# Calculates mAP@50 and F1 score on COCO val2017 dataset
# Includes object size stratification: small / medium / large
# COCO category ID mapping (COCO uses non-contiguous IDs)
# Map from COCO category ID to 0-indexed class ID
COCO_ID_TO_INDEX = {
    1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 10: 9,
    11: 10, 13: 11, 14: 12, 15: 13, 16: 14, 17: 15, 18: 16, 19: 17,
    20: 18, 21: 19, 22: 20, 23: 21, 24: 22, 25: 23, 27: 24, 28: 25,
    31: 26, 32: 27, 33: 28, 34: 29, 35: 30, 36: 31, 37: 32, 38: 33,
    39: 34, 40: 35, 41: 36, 42: 37, 43: 38, 44: 39, 46: 40, 47: 41,
    48: 42, 49: 43, 50: 44, 51: 45, 52: 46, 53: 47, 54: 48, 55: 49,
    56: 50, 57: 51, 58: 52, 59: 53, 60: 54, 61: 55, 62: 56, 63: 57,
    64: 58, 65: 59, 67: 60, 70: 61, 72: 62, 73: 63, 74: 64, 75: 65,
    76: 66, 77: 67, 78: 68, 79: 69, 80: 70, 81: 71, 82: 72, 84: 73,
    85: 74, 86: 75, 87: 76, 88: 77, 89: 78, 90: 79
}

# Reverse mapping: 0-indexed to COCO ID
INDEX_TO_COCO_ID = {v: k for k, v in COCO_ID_TO_INDEX.items()}

# COCO class names (80 classes)
COCO_CLASS_NAMES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck',
    'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench',
    'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra',
    'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee',
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup',
    'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange',
    'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
    'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse',
    'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush'
]

# =============================================================================
# Object Size Classification (COCO standard thresholds)
# =============================================================================
SIZE_THRESHOLDS = {
    'small': (0, 1024),
    'medium': (1024, 9216),
    'large': (9216, float('inf'))
}

def classify_size(area):
    """Classify an object as small, medium, or large based on COCO area thresholds."""
    if area < 1024:
        return 'small'
    elif area < 9216:
        return 'medium'
    else:
        return 'large'


class COCOEvaluator:
    """Evaluator for COCO val2017 dataset - calculates mAP@50 and F1 score."""
    
    def __init__(self, iou_threshold=0.5):
        self.iou_threshold = iou_threshold
        self.predictions = defaultdict(list)
        self.ground_truths = defaultdict(list)
        self.image_ids = []
    
    def reset(self):
        """Reset evaluator state."""
        self.predictions = defaultdict(list)
        self.ground_truths = defaultdict(list)
        self.image_ids = []

    def add_batch(self, image_id, pred_boxes, pred_scores, pred_classes, gt_boxes, gt_classes, gt_areas=None):
        """Add predictions and ground truths for an image."""
        if image_id not in self.image_ids:
            self.image_ids.append(image_id)
        
        for box, score, cls in zip(pred_boxes, pred_scores, pred_classes):
            self.predictions[cls].append({
                'image_id': image_id,
                'box': box,
                'score': score
            })
            
        for i, (box, cls) in enumerate(zip(gt_boxes, gt_classes)):
            gt_entry = {
                'image_id': image_id,
                'box': box,
                'matched': False
            }
            if gt_areas is not None:
                gt_entry['area'] = gt_areas[i]
                gt_entry['size'] = classify_size(gt_areas[i])
            self.ground_truths[cls].append(gt_entry)

    def compute_iou(self, box1, box2):
        """Compute IoU between two boxes in xyxy format."""
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        
        inter_area = max(0, x2 - x1) * max(0, y2 - y1)
        box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
        box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
        
        union_area = box1_area + box2_area - inter_area
        return inter_area / union_area if union_area > 0 else 0

    def _compute_ap_f1_for_subset(self, predictions_subset, gt_subset):
        """Compute AP and F1 for a subset of predictions and ground truths."""
        aps = []
        f1_scores = []
        class_results = []
        
        all_classes = set(predictions_subset.keys()) | set(gt_subset.keys())
        
        for cls in sorted(all_classes):
            preds = sorted(predictions_subset.get(cls, []), key=lambda x: x['score'], reverse=True)
            gts = gt_subset.get(cls, [])
            
            n_pos = len(gts)
            if n_pos == 0:
                continue
            
            tp = np.zeros(len(preds))
            fp = np.zeros(len(preds))
            
            gt_by_image = defaultdict(list)
            for gt in gts:
                gt['_matched'] = False
                gt_by_image[gt['image_id']].append(gt)
            
            for i, pred in enumerate(preds):
                image_id = pred['image_id']
                pred_box = pred['box']
                
                best_iou = 0
                best_gt = None
                
                candidates = gt_by_image[image_id]
                for gt in candidates:
                    if not gt['_matched']:
                        iou = self.compute_iou(pred_box, gt['box'])
                        if iou > best_iou:
                            best_iou = iou
                            best_gt = gt
                
                if best_iou >= self.iou_threshold:
                    tp[i] = 1
                    best_gt['_matched'] = True
                else:
                    fp[i] = 1
            
            tp_cumsum = np.cumsum(tp)
            fp_cumsum = np.cumsum(fp)
            recalls = tp_cumsum / n_pos if n_pos > 0 else tp_cumsum
            precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-6)
            
            mrec = np.concatenate(([0.0], recalls, [1.0]))
            mpre = np.concatenate(([0.0], precisions, [0.0]))
            
            for i in range(mpre.size - 1, 0, -1):
                mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
                
            i = np.where(mrec[1:] != mrec[:-1])[0]
            ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
            aps.append(ap)
            
            final_tp = np.sum(tp)
            final_fp = np.sum(fp)
            final_fn = n_pos - final_tp
            
            precision_scalar = final_tp / (final_tp + final_fp + 1e-6)
            recall_scalar = final_tp / (final_tp + final_fn + 1e-6)
            f1 = 2 * (precision_scalar * recall_scalar) / (precision_scalar + recall_scalar + 1e-6)
            f1_scores.append(f1)
            
            class_name = COCO_CLASS_NAMES[cls] if cls < len(COCO_CLASS_NAMES) else f"class_{cls}"
            class_results.append({
                'class': class_name, 'class_id': cls,
                'tp': int(final_tp), 'fp': int(final_fp), 'fn': int(final_fn),
                'ap': ap, 'f1': f1
            })
        
        mAP50 = np.mean(aps) if aps else 0.0
        mean_f1 = np.mean(f1_scores) if f1_scores else 0.0
        return mAP50, mean_f1, class_results

    def compute_metrics(self, verbose=True):
        """Compute mAP@50 and mean F1 score across all classes."""
        aps = []
        f1_scores = []
        class_results = []
        
        all_classes = set(self.predictions.keys()) | set(self.ground_truths.keys())
        
        for cls in sorted(all_classes):
            preds = sorted(self.predictions[cls], key=lambda x: x['score'], reverse=True)
            gts = self.ground_truths[cls]
            
            n_pos = len(gts)
            if n_pos == 0:
                continue
                
            tp = np.zeros(len(preds))
            fp = np.zeros(len(preds))
            
            # Group GTs by image
            gt_by_image = defaultdict(list)
            for gt in gts:
                gt['matched'] = False
                gt_by_image[gt['image_id']].append(gt)
            
            for i, pred in enumerate(preds):
                image_id = pred['image_id']
                pred_box = pred['box']
                
                best_iou = 0
                best_gt = None
                
                candidates = gt_by_image[image_id]
                for gt in candidates:
                    if not gt['matched']:
                        iou = self.compute_iou(pred_box, gt['box'])
                        if iou > best_iou:
                            best_iou = iou
                            best_gt = gt
                
                if best_iou >= self.iou_threshold:
                    tp[i] = 1
                    best_gt['matched'] = True
                else:
                    fp[i] = 1
            
            # Compute AP
            tp_cumsum = np.cumsum(tp)
            fp_cumsum = np.cumsum(fp)
            recalls = tp_cumsum / n_pos if n_pos > 0 else tp_cumsum
            precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-6)
            
            # 11-point interpolation
            mrec = np.concatenate(([0.0], recalls, [1.0]))
            mpre = np.concatenate(([0.0], precisions, [0.0]))
            
            for i in range(mpre.size - 1, 0, -1):
                mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
                
            i = np.where(mrec[1:] != mrec[:-1])[0]
            ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
            aps.append(ap)
            
            # Compute F1
            final_tp = np.sum(tp)
            final_fp = np.sum(fp)
            final_fn = n_pos - final_tp
            
            precision_scalar = final_tp / (final_tp + final_fp + 1e-6)
            recall_scalar = final_tp / (final_tp + final_fn + 1e-6)
            f1 = 2 * (precision_scalar * recall_scalar) / (precision_scalar + recall_scalar + 1e-6)
            f1_scores.append(f1)
            
            class_name = COCO_CLASS_NAMES[cls] if cls < len(COCO_CLASS_NAMES) else f"class_{cls}"
            class_results.append({
                'class': class_name,
                'class_id': cls,
                'tp': int(final_tp),
                'fp': int(final_fp),
                'fn': int(final_fn),
                'ap': ap,
                'f1': f1
            })
            
        mAP50 = np.mean(aps) if aps else 0.0
        mean_f1 = np.mean(f1_scores) if f1_scores else 0.0
        
        if verbose:
            print(f"\nPer-class results (top 10 by number of GT instances):")
            sorted_results = sorted(class_results, key=lambda x: x['tp'] + x['fn'], reverse=True)[:10]
            for r in sorted_results:
                print(f"  {r['class']:15s}: TP={r['tp']:4d}, FP={r['fp']:4d}, FN={r['fn']:4d}, AP={r['ap']:.4f}, F1={r['f1']:.4f}")
        
        return mAP50, mean_f1, class_results

    def compute_metrics_by_size(self, verbose=True):
        """Compute mAP@50 and F1 stratified by object size (small/medium/large)."""
        has_area = False
        for cls, gts in self.ground_truths.items():
            if gts and 'area' in gts[0]:
                has_area = True
                break
        
        if not has_area:
            print("WARNING: No area information available. Cannot compute size-stratified metrics.")
            return {}
        
        size_results = {}
        
        for size_name in ['small', 'medium', 'large']:
            gt_subset = defaultdict(list)
            gt_count = 0
            
            for cls, gts in self.ground_truths.items():
                for gt in gts:
                    if gt.get('size') == size_name:
                        gt_subset[cls].append({
                            'image_id': gt['image_id'], 'box': gt['box'],
                            'matched': False, 'area': gt['area'], 'size': gt['size']
                        })
                        gt_count += 1
            
            if gt_count == 0:
                size_results[size_name] = {'mAP50': 0.0, 'f1': 0.0, 'gt_count': 0, 'class_results': []}
                continue
            
            pred_subset = defaultdict(list)
            size_classes = set(gt_subset.keys())
            for cls in size_classes:
                pred_subset[cls] = [p for p in self.predictions.get(cls, [])]
            
            mAP50, mean_f1, class_results = self._compute_ap_f1_for_subset(pred_subset, gt_subset)
            size_results[size_name] = {
                'mAP50': mAP50, 'f1': mean_f1,
                'gt_count': gt_count, 'class_results': class_results
            }
        
        if verbose:
            total_gt = sum(sr['gt_count'] for sr in size_results.values())
            print(f"\n{'='*60}")
            print(f"SIZE-STRATIFIED RESULTS (COCO area thresholds)")
            print(f"{'='*60}")
            for sn, label in [('small', 'Small  (area < 32²)'), ('medium', 'Medium (32² ≤ area < 96²)'), ('large', 'Large  (area ≥ 96²)')]:
                sr = size_results[sn]
                pct = sr['gt_count']/total_gt*100 if total_gt > 0 else 0
                print(f"  {label}: {sr['gt_count']:>6d} objects ({pct:.1f}%) | "
                      f"mAP@50: {sr['mAP50']*100:.2f}% | F1: {sr['f1']*100:.2f}%")
            print(f"{'='*60}")
        
        return size_results


# Global cache for COCO annotations
_coco_val_cache = None

def load_coco_val_annotations():
    """Load and cache COCO val2017 annotations."""
    global _coco_val_cache
    if _coco_val_cache is not None:
        return _coco_val_cache
    
    ann_path = r"C:\desktop 2\project\datasets\COCO\annotations\instances_val2017.json"
    
    if not os.path.exists(ann_path):
        print(f"ERROR: COCO annotations not found at {ann_path}")
        return None
    
    print(f"Loading COCO val2017 annotations from: {ann_path}")
    
    with open(ann_path, 'r') as f:
        data = json.load(f)
    
    # Build image_id to filename mapping
    id_to_filename = {}
    filename_to_id = {}
    for img in data.get('images', []):
        img_id = img['id']
        filename = os.path.splitext(img['file_name'])[0]
        id_to_filename[img_id] = filename
        filename_to_id[filename] = img_id
    
    # Group annotations by image_id (area field is retained for size classification)
    annotations_by_image = defaultdict(list)
    for ann in data.get('annotations', []):
        # Skip crowd annotations for evaluation
        if ann.get('iscrowd', 0):
            continue
        annotations_by_image[ann['image_id']].append(ann)
    
    # Build category mapping
    categories = {cat['id']: cat['name'] for cat in data.get('categories', [])}
    
    _coco_val_cache = {
        'id_to_filename': id_to_filename,
        'filename_to_id': filename_to_id,
        'annotations_by_image': annotations_by_image,
        'categories': categories
    }
    
    total_annotations = sum(len(v) for v in annotations_by_image.values())
    print(f"Loaded {len(id_to_filename)} images, {total_annotations} annotations (non-crowd)")
    
    return _coco_val_cache


def load_coco_ground_truth(image_path):
    """Load ground truth for a COCO val2017 image.
    
    Returns:
        tuple: (boxes, classes, areas)
    """
    coco_data = load_coco_val_annotations()
    if coco_data is None:
        return [], [], []
    
    # Extract image ID from filename (e.g., "000000000139.jpg" -> 139)
    image_filename = os.path.basename(image_path)
    image_name_no_ext = os.path.splitext(image_filename)[0]
    
    # Try to get image_id
    image_id = coco_data['filename_to_id'].get(image_name_no_ext)
    if image_id is None:
        try:
            image_id = int(image_name_no_ext)
        except ValueError:
            return [], [], []
    
    annotations = coco_data['annotations_by_image'].get(image_id, [])
    
    boxes = []
    classes = []
    areas = []
    
    for ann in annotations:
        # COCO format: [x, y, width, height] -> [x1, y1, x2, y2]
        x, y, w, h = ann['bbox']
        x1, y1, x2, y2 = x, y, x + w, y + h
        
        # Convert COCO category ID to 0-indexed class ID
        coco_cat_id = ann['category_id']
        class_idx = COCO_ID_TO_INDEX.get(coco_cat_id)
        
        if class_idx is not None:
            boxes.append([x1, y1, x2, y2])
            classes.append(class_idx)
            areas.append(ann.get('area', w * h))
    
    return boxes, classes, areas


def run_coco_evaluation(max_images=None, verbose=True):
    """
    Run Mask2Former evaluation on COCO val2017 dataset.
    
    Returns:
        tuple: (mAP@50, mean_F1, class_results, size_results)
    """
    global _coco_val_cache
    _coco_val_cache = None  # Reset cache
    
    coco_path = r"C:\desktop 2\project\datasets\COCO\val2017"
    
    # Get all images
    image_files = glob.glob(os.path.join(coco_path, "*.jpg"))
    
    if max_images:
        image_files = image_files[:max_images]
    
    print(f"\n{'='*60}")
    print(f"COCO val2017 Evaluation - MASK2FORMER")
    print(f"{'='*60}")
    print(f"Images to evaluate: {len(image_files)}")
    
    evaluator = COCOEvaluator(iou_threshold=0.5)
    
    processed = 0
    images_with_gt = 0
    
    for idx, img_path in enumerate(image_files):
        gt_boxes, gt_classes, gt_areas = load_coco_ground_truth(img_path)
        
        if not gt_boxes:
            continue
        
        images_with_gt += 1
        
        pred_boxes = []
        pred_scores = []
        pred_classes = []
        
        image = PILImage.open(img_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = mask2former_model(**inputs)
        
        result = processor.post_process_panoptic_segmentation(
            outputs, target_sizes=[image.size[::-1]],
            label_ids_to_fuse=set(range(80, 133))
        )[0]
        
        panoptic_seg = result["segmentation"].cpu().numpy()
        segments_info = result["segments_info"]
        
        for seg in segments_info:
            label_id = seg['label_id']
            if label_id >= 80:  # Skip stuff classes
                continue
                
            seg_id = seg['id']
            score = seg.get('score', 1.0)
            
            if score < 0.5:  # Filter low confidence predictions
                continue  
            
            mask = panoptic_seg == seg_id
            if np.any(mask):
                y_coords, x_coords = np.where(mask)
                x_min, x_max = np.min(x_coords), np.max(x_coords)
                y_min, y_max = np.min(y_coords), np.max(y_coords)
                
                pred_boxes.append([x_min, y_min, x_max, y_max])
                pred_scores.append(score)
                pred_classes.append(label_id)
        
        evaluator.add_batch(img_path, pred_boxes, pred_scores, pred_classes, gt_boxes, gt_classes, gt_areas=gt_areas)
        processed += 1
        
        # Progress update every 100 images
        if (idx + 1) % 100 == 0:
            print(f"  Processed {idx + 1}/{len(image_files)} images...")
    
    print(f"\nImages processed: {processed}")
    print(f"Images with GT: {images_with_gt}")
    
    if images_with_gt == 0:
        print("ERROR: No images with ground truth found!")
        return 0.0, 0.0, [], {}
    
    mAP50, mean_f1, class_results = evaluator.compute_metrics(verbose=verbose)
    size_results = evaluator.compute_metrics_by_size(verbose=verbose)
    
    print(f"\n{'='*60}")
    print(f"COCO val2017 Results (Mask2Former):")
    print(f"  mAP@50: {mAP50:.4f} ({mAP50*100:.2f}%)")
    print(f"  Mean F1: {mean_f1:.4f} ({mean_f1*100:.2f}%)")
    print(f"{'='*60}")
    
    return mAP50, mean_f1, class_results, size_results


# Example usage:
print("COCO val2017 evaluation function loaded.")
print("Object size classification enabled (COCO thresholds: small<1024, medium<9216, large>=9216 px²)")

In [ ]:
# Evaluate Mask2Former on all COCO Haze Datasets
# mAP@50 and F1 scores for intensities 0.5, 1.0, 1.5

import pandas as pd
import torch
from PIL import Image as PILImage

def run_coco_evaluation_custom_path(image_dir=None, max_images=None, verbose=True):
    """
    Run Mask2Former evaluation on COCO images from a custom directory.
    Uses original COCO annotations but images from specified directory.
    
    Returns:
        tuple: (mAP@50, mean_F1, class_results, size_results)
    """
    global _coco_val_cache
    _coco_val_cache = None  # Reset cache
    
    if image_dir is None:
        image_dir = r"C:\desktop 2\project\datasets\COCO\val2017"
    
    # Get all images from the specified directory
    image_files = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))
    
    if max_images:
        image_files = image_files[:max_images]
    
    print(f"\n{'='*60}")
    print(f"COCO Evaluation - MASK2FORMER")
    print(f"{'='*60}")
    print(f"Image directory: {image_dir}")
    print(f"Images to evaluate: {len(image_files)}")
    
    evaluator = COCOEvaluator(iou_threshold=0.5)
    
    processed = 0
    images_with_gt = 0
    
    for idx, img_path in enumerate(image_files):
        gt_boxes, gt_classes, gt_areas = load_coco_ground_truth(img_path)
        
        if not gt_boxes:
            continue
        
        images_with_gt += 1
        
        pred_boxes = []
        pred_scores = []
        pred_classes = []
        
        image = PILImage.open(img_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = mask2former_model(**inputs)
        
        result = processor.post_process_panoptic_segmentation(
            outputs, target_sizes=[image.size[::-1]],
            label_ids_to_fuse=set(range(80, 133))
        )[0]
        
        panoptic_seg = result["segmentation"].cpu().numpy()
        segments_info = result["segments_info"]
        
        for seg in segments_info:
            label_id = seg['label_id']
            if label_id >= 80:  # Skip stuff classes
                continue
                
            seg_id = seg['id']
            score = seg.get('score', 1.0)
            
            if score < 0.5:  # Filter low-confidence segments
                continue
            
            mask = panoptic_seg == seg_id
            if np.any(mask):
                y_coords, x_coords = np.where(mask)
                x_min, x_max = np.min(x_coords), np.max(x_coords)
                y_min, y_max = np.min(y_coords), np.max(y_coords)
                
                pred_boxes.append([x_min, y_min, x_max, y_max])
                pred_scores.append(score)
                pred_classes.append(label_id)
        
        evaluator.add_batch(img_path, pred_boxes, pred_scores, pred_classes, gt_boxes, gt_classes, gt_areas=gt_areas)
        processed += 1
        
        # Progress update every 500 images
        if (idx + 1) % 500 == 0:
            print(f"  Processed {idx + 1}/{len(image_files)} images...")
    
    print(f"\nImages processed: {processed}")
    print(f"Images with GT: {images_with_gt}")
    
    if images_with_gt == 0:
        print("ERROR: No images with ground truth found!")
        return 0.0, 0.0, [], {}
    
    mAP50, mean_f1, class_results = evaluator.compute_metrics(verbose=verbose)
    size_results = evaluator.compute_metrics_by_size(verbose=verbose)
    
    print(f"\n{'='*60}")
    print(f"Results (Mask2Former):")
    print(f"  mAP@50: {mAP50:.4f} ({mAP50*100:.2f}%)")
    print(f"  Mean F1: {mean_f1:.4f} ({mean_f1*100:.2f}%)")
    print(f"{'='*60}")
    
    return mAP50, mean_f1, class_results, size_results

# Dataset paths
haze_datasets = {
    "Haze 0.5": r"C:\desktop 2\project\datasets\COCO\COCO_haze_0.5",
    "Haze 1.0": r"C:\desktop 2\project\datasets\COCO\COCO_haze_1.0",
    "Haze 1.5": r"C:\desktop 2\project\datasets\COCO\COCO_haze_1.5"
}

# Store all results (including size-stratified)
all_results = []
all_size_results = {}

# Evaluate on each haze dataset
for dataset_name, dataset_path in haze_datasets.items():
    print(f"\n{'#'*70}")
    print(f"# EVALUATING ON: {dataset_name}")
    print(f"# Path: {dataset_path}")
    print(f"{'#'*70}")
    
    # Evaluate Mask2Former
    print(f"\n>>> Mask2Former-Swin-Large on {dataset_name}")
    m2f_map, m2f_f1, _, m2f_size = run_coco_evaluation_custom_path(
        image_dir=dataset_path, verbose=False
    )
    all_results.append({
        'Dataset': dataset_name,
        'Model': 'Mask2Former-Swin-Large',
        'mAP@50': m2f_map,
        'F1': m2f_f1
    })
    all_size_results[(dataset_name, 'Mask2Former-Swin-Large')] = m2f_size

# Create results DataFrame
results_df = pd.DataFrame(all_results)

# Pivot table for better visualization
print(f"\n{'='*80}")
print("EVALUATION RESULTS SUMMARY - COCO Haze Datasets")
print(f"{'='*80}")

# Create pivot tables for mAP@50 and F1
map_pivot = results_df.pivot(index='Model', columns='Dataset', values='mAP@50')
f1_pivot = results_df.pivot(index='Model', columns='Dataset', values='F1')

# Reorder columns
col_order = ['Haze 0.5', 'Haze 1.0', 'Haze 1.5']
map_pivot = map_pivot[col_order]
f1_pivot = f1_pivot[col_order]

print("\n" + "="*50)
print("mAP@50 Scores (%)")
print("="*50)
map_percent = map_pivot * 100
print(map_percent.round(2).to_string())

print("\n" + "="*50)
print("F1 Scores (%)")
print("="*50)
f1_percent = f1_pivot * 100
print(f1_percent.round(2).to_string())

# Combined table
print("\n" + "="*80)
print("COMBINED RESULTS TABLE")
print("="*80)
combined_df = results_df.copy()
combined_df['mAP@50 (%)'] = (combined_df['mAP@50'] * 100).round(2)
combined_df['F1 (%)'] = (combined_df['F1'] * 100).round(2)
combined_df = combined_df[['Dataset', 'Model', 'mAP@50 (%)', 'F1 (%)']]
print(combined_df.to_string(index=False))

# Display as formatted table
print("\n" + "="*80)
print("FORMATTED RESULTS TABLE")
print("="*80)
print(f"\n{'Model':<25} {'Dataset':<12} {'mAP@50':>12} {'F1':>12}")
print("-" * 65)
for _, row in combined_df.iterrows():
    print(f"{row['Model']:<25} {row['Dataset']:<12} {row['mAP@50 (%)']:>11.2f}% {row['F1 (%)']:>11.2f}%")
print("="*80)

# =============================================================================
# SIZE-STRATIFIED RESULTS - HAZE DATASETS
# =============================================================================
print(f"\n{'='*80}")
print("SIZE-STRATIFIED RESULTS - COCO Haze Datasets")
print(f"{'='*80}")

size_rows = []
for (dataset_name, model_name), sr in all_size_results.items():
    for size_name in ['small', 'medium', 'large']:
        if size_name in sr:
            size_rows.append({
                'Dataset': dataset_name,
                'Model': model_name,
                'Size': size_name.capitalize(),
                'GT Count': sr[size_name]['gt_count'],
                'mAP@50 (%)': round(sr[size_name]['mAP50'] * 100, 2),
                'F1 (%)': round(sr[size_name]['f1'] * 100, 2)
            })

if size_rows:
    size_df = pd.DataFrame(size_rows)
    print(f"\n{'Model':<25} {'Dataset':<12} {'Size':<8} {'GT Count':>10} {'mAP@50':>12} {'F1':>12}")
    print("-" * 85)
    for _, row in size_df.iterrows():
        print(f"{row['Model']:<25} {row['Dataset']:<12} {row['Size']:<8} {row['GT Count']:>10} "
              f"{row['mAP@50 (%)']:>11.2f}% {row['F1 (%)']:>11.2f}%")
    print("="*80)

In [ ]:
# For full COCO val2017 (5000 images):
m2f_map, m2f_f1, m2f_results, m2f_size_results = run_coco_evaluation()

In [ ]:
# =============================================================================
# Mask2Former Performance by Object Size Class — Results Table
# =============================================================================
# Shows Mask2Former's mAP@50 and F1 success rate per size class (small/medium/large)
# across clean and all haze/degraded conditions.

import pandas as pd

# --- Collect all available size-stratified results ---
all_model_size_data = []

# 1. Clean COCO val2017 results
if 'm2f_size_results' in globals() and m2f_size_results:
    for size_name in ['small', 'medium', 'large']:
        if size_name in m2f_size_results:
            s = m2f_size_results[size_name]
            all_model_size_data.append({
                'Condition': 'Clean',
                'Model': 'Mask2Former',
                'Size Class': size_name.capitalize(),
                'GT Objects': s['gt_count'],
                'mAP@50 (%)': round(s['mAP50'] * 100, 2),
                'F1 (%)': round(s['f1'] * 100, 2)
            })

# 2. Haze results
if 'all_size_results' in globals() and all_size_results:
    for (dataset_name, model_name), sr in all_size_results.items():
        for size_name in ['small', 'medium', 'large']:
            if size_name in sr:
                s = sr[size_name]
                all_model_size_data.append({
                    'Condition': dataset_name,
                    'Model': model_name,
                    'Size Class': size_name.capitalize(),
                    'GT Objects': s['gt_count'],
                    'mAP@50 (%)': round(s['mAP50'] * 100, 2),
                    'F1 (%)': round(s['f1'] * 100, 2)
                })

# 3. Degraded results
if 'degraded_size_results' in globals() and degraded_size_results:
    for model_name, sr in degraded_size_results.items():
        for size_name in ['small', 'medium', 'large']:
            if size_name in sr:
                s = sr[size_name]
                all_model_size_data.append({
                    'Condition': 'Degraded',
                    'Model': model_name,
                    'Size Class': size_name.capitalize(),
                    'GT Objects': s['gt_count'],
                    'mAP@50 (%)': round(s['mAP50'] * 100, 2),
                    'F1 (%)': round(s['f1'] * 100, 2)
                })

if not all_model_size_data:
    print("No size-stratified results available yet. Run the evaluation cells first.")
else:
    df = pd.DataFrame(all_model_size_data)

    # =========================================================================
    # TABLE 1: Per-condition success by size class (pivot: Size as columns)
    # =========================================================================
    print("=" * 90)
    print("MASK2FORMER PERFORMANCE BY OBJECT SIZE CLASS")
    print("=" * 90)
    print(f"\n{'Condition':<14} {'Model':<22} {'Metric':<10} {'Small':>10} {'Medium':>10} {'Large':>10}")
    print("-" * 80)

    # Group by condition and model
    for condition in df['Condition'].unique():
        cond_df = df[df['Condition'] == condition]
        for model_name in cond_df['Model'].unique():
            model_df = cond_df[cond_df['Model'] == model_name]

            # Build lookup: size -> metrics
            size_map = {}
            for _, row in model_df.iterrows():
                size_map[row['Size Class']] = row

            small_map = size_map.get('Small', {})
            med_map = size_map.get('Medium', {})
            large_map = size_map.get('Large', {})

            s_map = small_map.get('mAP@50 (%)', 0) if isinstance(small_map, dict) else small_map['mAP@50 (%)']
            m_map = med_map.get('mAP@50 (%)', 0) if isinstance(med_map, dict) else med_map['mAP@50 (%)']
            l_map = large_map.get('mAP@50 (%)', 0) if isinstance(large_map, dict) else large_map['mAP@50 (%)']

            s_f1 = small_map.get('F1 (%)', 0) if isinstance(small_map, dict) else small_map['F1 (%)']
            m_f1 = med_map.get('F1 (%)', 0) if isinstance(med_map, dict) else med_map['F1 (%)']
            l_f1 = large_map.get('F1 (%)', 0) if isinstance(large_map, dict) else large_map['F1 (%)']

            print(f"{condition:<14} {model_name:<22} {'mAP@50':>10} {s_map:>9.2f}% {m_map:>9.2f}% {l_map:>9.2f}%")
            print(f"{'':14} {'':22} {'F1':>10} {s_f1:>9.2f}% {m_f1:>9.2f}% {l_f1:>9.2f}%")
            print("-" * 80)

    # =========================================================================
    # TABLE 2: Pivot table using pandas for clean presentation
    # =========================================================================
    print("\n" + "=" * 90)
    print("PIVOT TABLE — mAP@50 (%) BY CONDITION × SIZE CLASS")
    print("=" * 90)

    pivot_map = df.pivot_table(
        index=['Model', 'Condition'],
        columns='Size Class',
        values='mAP@50 (%)',
        aggfunc='first'
    )
    # Reorder columns
    for col in ['Small', 'Medium', 'Large']:
        if col not in pivot_map.columns:
            pivot_map[col] = 0.0
    pivot_map = pivot_map[['Small', 'Medium', 'Large']]
    print(pivot_map.round(2).to_string())

    print("\n" + "=" * 90)
    print("PIVOT TABLE — F1 (%) BY CONDITION × SIZE CLASS")
    print("=" * 90)

    pivot_f1 = df.pivot_table(
        index=['Model', 'Condition'],
        columns='Size Class',
        values='F1 (%)',
        aggfunc='first'
    )
    for col in ['Small', 'Medium', 'Large']:
        if col not in pivot_f1.columns:
            pivot_f1[col] = 0.0
    pivot_f1 = pivot_f1[['Small', 'Medium', 'Large']]
    print(pivot_f1.round(2).to_string())
